# Проверка загрузки данных и токенизации (Фаза 1)
Этот ноутбук предназначен для локальной верификации системы данных BEBLaDII перед запуском обучения в Kaggle.

In [1]:
%load_ext autoreload
%autoreload 2
    
import os
import sys
# Добавляем корень проекта в пути
sys.path.append(os.path.abspath('..'))

import torch
from src.beb_la_dii.utils.data import get_dataloader
from src.beb_la_dii.utils.tokenizer import get_tokenizer
from indexed_parquet_dataset import IndexedParquetDataset

print("Библиотеки загружены успешно.")

C:\Experiments\BEBLaDII\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Библиотеки загружены успешно.


## 1. Проверка сырых данных
Посмотрим, видит ли `IndexedParquetDataset` наши файлы.

In [2]:
paths = ['data/CulturaX', 'data/magpie_reasoning', 'data/open_thoughts']
for p in paths:
    path = os.path.join('..', p)
    if os.path.exists(path):
        ds = IndexedParquetDataset.from_folder(path, auto_fill=True)
        print(f"Dataset {p}: {len(ds)} примеров.")
    else:
        print(f"Path not found: {path}")

Dataset data/CulturaX: 1201108 примеров.
Dataset data/magpie_reasoning: 249922 примеров.
Dataset data/open_thoughts: 227914 примеров.


## 2. Проверка DataLoader (Stage: Awakening)
Проверим балансировку 90/10.

In [3]:
# Сделаем небольшой хак с путями для теста внутри ноутбука
os.chdir('..')
loader = get_dataloader(stage='awakening', batch_size=2)

for i, batch in enumerate(loader):
    print(f"Batch {i} input_ids shape: {batch['input_ids'].shape}")
    # Декодируем один пример
    tokenizer = get_tokenizer()
    text = tokenizer.decode(batch['input_ids'][0], skip_special_tokens=False)
    print("--- Sample text ---")
    print(text[:500] + "...")
    break

Инициализирован комбинированный датасет: 100000 примеров.
Batch 0 input_ids shape: torch.Size([2, 512])
--- Sample text ---
<|im_start|>user
Given an unrooted tree with N nodes, how many ways are there to assign each node a color (let's say red or blue) such that no two adjacent nodes have the same color?<|im_end|>
<|im_start|>assistant
<|thought|>
So I have this problem here: given an unrooted tree with N nodes, I need to figure out how many ways there are to color each node either red or blue, but with the condition that no two adjacent nodes can have the same color. Okay, let's break this down.

First off, what's ...
